In [6]:
# magics: ensures that any changes to the modules loaded below will be re-loaded automatically
%load_ext autoreload
%autoreload 2

# load general packages
import os
os.chdir('/Users/jacobaspnissen/Desktop/Økonomi/Dynamic programming/Termpaper/termpaper_dynprog')
import numpy as np
import time
import copy
import pandas as pd
import pyreadr
import pickle
import xarray as xr  # Ensure xarray is installed



import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
plt.style.use('seaborn-v0_8-whitegrid')

# load modules related EGM
#import tools 
#from model_exante import model_bufferstock
#import estimate_exante as estimate

# load modules related to NFXP
from model_retirement import retirement
from Solve_NFXP import solve_NFXP
import estimate_NFXP as estimate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Import Data 

In [19]:
# Clean data
cleaned_lines = []
with open("KPS Data/sparadata2.txt", "r") as file:
    for line in file:
        # Replace multiple spaces with a single tab
        cleaned_line = " ".join(line.split())  # Normalize spaces
        cleaned_lines.append(cleaned_line)

# Save the cleaned file
with open("KPS Data/cleaned_sparadata.txt", "w") as cleaned_file:
    cleaned_file.write("\n".join(cleaned_lines))

In [20]:
datapath = "KPS Data/cleaned_sparadata.txt"
sparadata = np.genfromtxt(open(datapath, "rb"), delimiter=" ", skip_header=0, dtype=None, encoding=None)


- index   is an identifier for the observation (each individual and year)
- id2    	is an identifier for the individual
- sex    	1=man 2=woman
- year   	of observation
- age    	of the individual for the given year
- married	marital status
- retire	yes=1, no=0
- income	1/1000 SEK. This number is rounded (no decimal digits) due to
	confidentiality reasons
- atp	is the "average pension points" used in the paper

In [25]:
data = pd.read_csv(datapath, sep=" ", header=0, encoding='utf-8')

In [27]:
data = data.drop(columns="index")

In [29]:
# remove all observations where age < 50
data2 = data[data['age'] >= 50]

In [36]:
dat = np.loadtxt(open(datapath), delimiter=" ", skiprows=1)

In [37]:
dat

array([[1.00000000e+00, 3.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 1.79000000e+02, 4.63600000e+00],
       [2.00000000e+00, 3.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 1.88000000e+02, 4.63600000e+00],
       [3.00000000e+00, 3.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 1.86000000e+02, 4.63600000e+00],
       ...,
       [6.66940000e+04, 4.40940000e+04, 2.00000000e+00, ...,
        0.00000000e+00, 1.54000000e+02, 2.94466667e+00],
       [6.66950000e+04, 4.40940000e+04, 2.00000000e+00, ...,
        0.00000000e+00, 1.54000000e+02, 3.00933333e+00],
       [6.66960000e+04, 4.40940000e+04, 2.00000000e+00, ...,
        0.00000000e+00, 1.54000000e+02, 3.06466667e+00]], shape=(51371, 9))

In [38]:
dat.shape

(51371, 9)

In [34]:
data

,id,sex,year,age,married,ret,income,atp
0,3,1,83,52,1,0.0,179,4.636000
1,3,1,84,53,1,0.0,188,4.636000
2,3,1,85,54,1,0.0,186,4.636000
3,3,1,86,55,1,0.0,189,4.636000
4,3,1,87,56,1,0.0,196,4.640667
...,...,...,...,...,...,...,...,...
51366,44094,2,92,52,1,0.0,154,2.803333
51367,44094,2,93,53,1,0.0,154,2.875333
51368,44094,2,94,54,0,0.0,154,2.944667
51369,44094,2,95,55,0,0.0,154,3.009333


In [30]:
data2

,id,sex,year,age,married,ret,income,atp
0,3,1,83,52,1,0.0,179,4.636000
1,3,1,84,53,1,0.0,188,4.636000
2,3,1,85,54,1,0.0,186,4.636000
3,3,1,86,55,1,0.0,189,4.636000
4,3,1,87,56,1,0.0,196,4.640667
...,...,...,...,...,...,...,...,...
51366,44094,2,92,52,1,0.0,154,2.803333
51367,44094,2,93,53,1,0.0,154,2.875333
51368,44094,2,94,54,0,0.0,154,2.944667
51369,44094,2,95,55,0,0.0,154,3.009333
